# BR 트렌드 레이더: 13주 생존 예측 모델

**질문:** 지금 검색량이 급등한 키워드가 13주 뒤(BR 이달의 맛 출시 리드타임)에도 살아있을까?

- **A. 지도학습:** 급등이 감지된 시점의 곡선만 보고 13주 뒤 생존 여부를 예측
- **B. 비지도학습:** 라벨 없이 곡선 모양만으로 군집을 나눠서, 우리가 정한 3유형(단발형/완만형/반복형)이 저절로 나오는지 확인

데이터: 네이버 데이터랩 주간 검색량 (2016.1 ~ 2026.10), 키워드 79개

## 0. 준비

In [ ]:
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # 그래프 한글 깨짐 방지
import pandas as pd, numpy as np, glob, warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

## 1. 데이터 불러오기
아래 셀을 실행하면 파일 선택 버튼이 나와. CSV 3개를 한 번에 선택해서 올리면 돼.
- `datalab_weekly.csv` (트렌드 25개)
- `datalab_ingredients_weekly.csv` (재료 21개)
- `datalab_br_ingredients_weekly.csv` (이달의 맛 재료 48개)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
frames = []
for f in sorted(glob.glob('datalab*.csv')):
    d = pd.read_csv(f, encoding='utf-8-sig')
    d.columns = ['kw', 'date', 'v']
    frames.append(d)
df = pd.concat(frames)
df['date'] = pd.to_datetime(df['date'])
df = df.drop_duplicates(['kw', 'date'])          # 여러 파일에 겹치는 키워드는 하나만

# 주 단위 표로 펼치기 (행=주, 열=키워드). 검색량이 0인 주는 API가 빼고 주기 때문에 0으로 채움
grid = pd.date_range(df.date.min(), df.date.max(), freq='W-MON')
W = df.pivot_table(index='date', columns='kw', values='v').reindex(grid).fillna(0)
print(f'기간 {W.index[0].date()} ~ {W.index[-1].date()}, {len(W)}주, 키워드 {W.shape[1]}개')

In [ ]:
# 예시: 성격이 다른 키워드 4개
ex = [k for k in ['탕후루', '두바이 초콜릿', '말차', '딸기'] if k in W.columns]
fig, axes = plt.subplots(len(ex), 1, figsize=(10, 2.2 * len(ex)), sharex=True)
for ax, k in zip(np.atleast_1d(axes), ex):
    ax.plot(W.index, W[k], lw=1); ax.set_title(k, loc='left', fontsize=11)
plt.tight_layout(); plt.show()

## 2. 학습 데이터 만들기

**한 줄 = "레이더가 급등을 감지한 어느 한 주"**
- 감지 조건: 이번 주 검색량이 직전 12주 평균의 1.5배 이상 (그리고 너무 작은 값은 제외)
- 같은 급등이 매주 반복해서 잡히지 않도록, 키워드마다 8주에 한 번만 사용

**정답(y):** 13주 뒤 검색량이 감지 시점의 50% 이상이면 1(버팀), 아니면 0(못 버팀)

**특징(X):** 감지 시점까지의 정보만 사용 (미래 정보는 절대 안 씀)
| 특징 | 뜻 |
|---|---|
| rise4 | 최근 4주 평균 대비 몇 배로 뛰었나 (급하게 떴나) |
| rise13 | 3개월 전 대비 몇 배인가 |
| rise_weeks | 지금 수준의 25% 이상으로 몇 주째 이어져 왔나. 짧으면 갑자기 튄 것 (우리 규칙의 핵심 변수) |
| pos_runmax | 지금까지 최고점 대비 현재 위치 |
| base52 | 지난 1년 평균이 현재 대비 얼마나 되나 (원래 찾던 사람이 있던 키워드인가) |
| lastyear | 작년 이맘때도 튀었나 (반복형 신호) |
| vol8 | 최근 8주 들쭉날쭉한 정도 |
| age | 처음 검색되기 시작한 지 몇 주 됐나 |

In [ ]:
H = 13   # BR 리드타임

def features(v, t):
    vt = v[t]
    k = 0
    while t - k - 1 >= 0 and v[t - k - 1] >= 0.25 * vt:
        k += 1
    prev_year = v[t-104:t-52].mean() if t >= 104 else v[:t-52].mean()
    return dict(
        rise4=np.log(vt / max(v[t-4:t].mean(), 0.5)),
        rise13=np.log(vt / max(v[t-13:t-9].mean(), 0.5)),
        rise_weeks=k,
        pos_runmax=vt / v[:t+1].max(),
        base52=v[t-52:t].mean() / vt,
        lastyear=np.log(max(v[t-56:t-47].max(), 0.5) / max(prev_year, 0.5)),
        vol8=np.std(np.diff(np.log(v[t-8:t+1] + 1))),
        age=t - np.argmax(v > 1) if (v > 1).any() else 0,
    )

def is_surge(v, t):
    return v[t] >= 5 and v[t] >= 1.5 * max(v[t-12:t].mean(), 0.5)

rows = []
for kw in W.columns:
    v = W[kw].values
    last = -99
    for t in range(60, len(v) - H):
        if not is_surge(v, t) or t - last < 8:
            continue
        last = t
        r = features(v, t)
        r.update(kw=kw, date=W.index[t], y=int(v[t + H] >= 0.5 * v[t]))
        rows.append(r)

S = pd.DataFrame(rows)
FEATS = ['rise4', 'rise13', 'rise_weeks', 'pos_runmax', 'base52', 'lastyear', 'vol8', 'age']
print(f'학습 샘플 {len(S)}개, 키워드 {S.kw.nunique()}개, 버팀 비율 {S.y.mean():.0%}')
S.head()

## 3. 모델 학습과 검증

**중요:** 같은 키워드가 학습과 시험에 동시에 들어가면 모델이 키워드를 외워버려.
그래서 **키워드 단위로 5묶음을 나눠** (GroupKFold), 한 번도 본 적 없는 키워드로 시험해.

비교 기준(베이스라인)은 지금 쓰는 규칙: `뜨는 데 5주 이하 → 못 버팀`

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score

X, y, groups = S[FEATS], S.y.values, S.kw.values
models = {
    '로지스틱 회귀': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced')),
    '랜덤 포레스트': RandomForestClassifier(300, min_samples_leaf=5, class_weight='balanced', random_state=0),
    '그래디언트 부스팅': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                                   class_weight='balanced', random_state=0),
}

results, oof = [], {}
for name, m in models.items():
    p = np.zeros(len(y))
    for tr, te in GroupKFold(5).split(X, y, groups):
        m.fit(X.iloc[tr], y[tr])
        p[te] = m.predict_proba(X.iloc[te])[:, 1]
    oof[name] = p
    results.append([name, accuracy_score(y, p > .5), balanced_accuracy_score(y, p > .5), roc_auc_score(y, p)])

rule = (S.rise_weeks > 5).astype(int)
results.append(['규칙 (뜨는 데 5주 이하면 못 버팀)', accuracy_score(y, rule), balanced_accuracy_score(y, rule), np.nan])

R = pd.DataFrame(results, columns=['모델', '정확도', '균형 정확도', 'AUC']).round(3)
R

**읽는 법**
- **정확도:** 전체 중 맞힌 비율. 버팀이 65%라서, 전부 "버팀"이라고 찍어도 65%가 나와.
- **균형 정확도:** 버팀과 못 버팀을 각각 얼마나 잘 맞혔는지 평균. 찍기는 50%. **이 숫자로 비교하는 게 공정해.**
- **AUC:** 버티는 키워드에 더 높은 점수를 줄 확률. 0.5는 동전 던지기, 1.0은 완벽.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(R['모델'], R['균형 정확도'], color=['#4C78A8'] * 3 + ['#BBBBBB'])
ax.axvline(0.5, ls='--', c='gray', lw=1); ax.text(0.505, -0.6, '찍기', color='gray', fontsize=9)
for i, val in enumerate(R['균형 정확도']):
    ax.text(val + 0.005, i, f'{val:.2f}', va='center', fontsize=9)
ax.set_xlim(0.4, 0.85); ax.set_xlabel('균형 정확도 (처음 보는 키워드 기준)'); ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 4. 모델은 무엇을 보고 판단했나
특징 하나를 마구 섞어버렸을 때 성능이 얼마나 떨어지는지로 중요도를 측정 (permutation importance).
많이 떨어질수록 모델이 그 특징에 많이 의존한다는 뜻.

In [ ]:
from sklearn.inspection import permutation_importance
best = models['랜덤 포레스트'].fit(X, y)
imp = permutation_importance(best, X, y, scoring='roc_auc', n_repeats=20, random_state=0)
I = pd.Series(imp.importances_mean, index=FEATS).sort_values()
names = {'rise4': '최근 4주 급등 폭', 'rise13': '3개월 전 대비', 'rise_weeks': '꾸준히 검색된 주 수',
         'pos_runmax': '역대 최고 대비 위치', 'base52': '원래 찾던 사람 (1년 평균)', 'lastyear': '작년 이맘때도 튐',
         'vol8': '들쭉날쭉함', 'age': '검색 시작 후 기간'}
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh([names[i] for i in I.index], I.values, color='#4C78A8')
ax.set_xlabel('섞었을 때 AUC 하락폭'); plt.tight_layout(); plt.show()

In [ ]:
# 특징 하나씩만 따로 봤을 때 방향 (AUC > 0.5: 클수록 버팀, < 0.5: 클수록 못 버팀)
pd.Series({names[f]: roc_auc_score(y, S[f]) for f in FEATS}).round(3).sort_values()

## 5. 지금 시점 예측
최근 26주 안에 급등이 감지된 키워드에 대해, 감지 시점 기준 13주 생존 확률을 계산.
(감지 후 13주가 아직 안 지난 건 진짜 예측이고, 지난 건 실제 결과와 비교 가능)

In [ ]:
final = models['랜덤 포레스트'].fit(X, y)
n = len(W)
now = []
for kw in W.columns:
    v = W[kw].values
    hits = [t for t in range(n - 26, n) if is_surge(v, t)]
    if not hits:
        continue
    t = hits[-1]
    r = features(v, t)
    prob = final.predict_proba(pd.DataFrame([r])[FEATS])[0, 1]
    actual = ('아직 모름' if t + H >= n else ('버팀' if v[t + H] >= 0.5 * v[t] else '못 버팀'))
    now.append(dict(키워드=kw, 감지주=W.index[t].date(), 생존확률=round(prob, 2),
                    꾸준히검색된주=r['rise_weeks'], 실제결과=actual))
NOW = pd.DataFrame(now).sort_values('생존확률', ascending=False).reset_index(drop=True)
NOW

## 6. B. 비지도학습: 곡선 모양으로 군집 나누기
라벨 없이, 각 키워드의 **최고점 앞 26주 ~ 뒤 52주 곡선 모양**과 **계절 반복 정도**만 넣고 K-means로 4묶음으로 나눔.
그다음 우리가 손으로 정한 규칙 유형(단발형/완만형/반복형)과 얼마나 겹치는지 확인.

In [ ]:
from sklearn.cluster import KMeans

def rule_type(v, dates):
    s = pd.Series(v, index=dates)
    ratio = (s / s.rolling(52, min_periods=26).mean().replace(0, np.nan)).dropna()
    ac = ratio.autocorr(52) if len(ratio) > 110 else 0           # 1년 주기 자기상관
    yr = s[(s.index.year >= 2016) & (s.index.year <= 2025)]
    pm = yr.groupby(yr.index.year).idxmax().dt.month               # 해마다 피크가 온 달
    mode = pm.mode().iloc[0]
    same = (np.minimum((pm - mode) % 12, (mode - pm) % 12) <= 1).mean()   # 같은 철(±1달) 비율
    p = int(np.argmax(v)); k = 0
    while p - k - 1 >= 0 and v[p - k - 1] >= 0.25 * v[p]:
        k += 1
    if ac >= 0.5 and same >= 0.8:
        typ = '반복형'
    else:
        typ = '단발형' if k <= 5 else ('완만형' if k >= 10 else '보류')
    return typ, ac, same

info, vecs = [], []
for kw in W.columns:
    v = W[kw].values
    typ, ac, same = rule_type(v, W.index)
    p = int(np.argmax(v))
    shape = np.array([v[i] if 0 <= i < len(v) else 0 for i in range(p - 26, p + 53)]) / v[p]
    vecs.append(np.r_[shape, ac * 2, same * 2])     # 모양 79칸 + 계절성 2칸
    info.append(dict(kw=kw, 규칙유형=typ))

C = pd.DataFrame(info)
V = np.array(vecs)
km = KMeans(4, n_init=20, random_state=0).fit(V)
C['군집'] = km.labels_
ct = pd.crosstab(C['군집'], C['규칙유형'])
ct

In [ ]:
# 군집별 평균 곡선 모양
fig, axes = plt.subplots(1, 4, figsize=(14, 2.8), sharey=True)
xs = np.arange(-26, 53)
for c, ax in enumerate(axes):
    members = V[km.labels_ == c, :79]
    ax.plot(xs, members.T, c='#4C78A8', alpha=0.08)
    ax.plot(xs, members.mean(0), c='#E45756', lw=2)
    top = ct.loc[c].idxmax()
    ax.set_title(f'군집 {c} ({len(members)}개)\n대부분 {top}', fontsize=10)
    ax.axvline(0, c='gray', lw=0.6); ax.set_xlabel('최고점 기준 주')
plt.tight_layout(); plt.show()

purity = ct.max(axis=1).sum() / ct.values.sum()
print(f'군집과 규칙 유형 일치율: {purity:.0%}')
for c in range(4):
    print(f'군집 {c}:', ', '.join(C[C.군집 == c].kw.head(8)))

## 7. 결과 저장 (앱 연결용)

In [ ]:
C.merge(NOW, left_on='kw', right_on='키워드', how='left').to_csv('radar_model_output.csv', index=False, encoding='utf-8-sig')
R.to_csv('model_scores.csv', index=False, encoding='utf-8-sig')
files.download('radar_model_output.csv')